<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 21
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Big Data - Machine Learning - Clustering (K-means)
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

# Create SparkSession

In [ ]:
from pcamarillor.spark_utils import SparkUtils
su = SparkUtils("ML: K-means", 
                "local[*]")
su.spark

# Example 1: Clustering with 2D points

In [ ]:
# Sample data in Python (e.g., 2D points)
data = [
    (0, 1.0, 1.0),
    (1, 2.0, 1.0),
    (2, 4.0, 5.0),
    (3, 5.0, 5.0),
    (4, 10.0, 10.0),
    (5, 12.0, 11.0)
]

# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("id", "int"), ("x", "float"), ("y", "float")])

# Create DataFrame for k means
random_points_df = su.spark.createDataFrame(data, schema)

## Assemble the features into a single vector column

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["x", "y"], outputCol="features")
assembled_df = assembler.transform(random_points_df)

In [ ]:
assembled_df.show()

## Configure K-means

In [ ]:
from pyspark.ml.clustering import KMeans

kmeans = KMeans().setK(3).setSeed(73)

## Train model

In [ ]:
model = kmeans.fit(assembled_df)
predictions = model.transform(assembled_df)
print("K-means model trained successfully")
kmeans_model_path = "/opt/spark/work-dir/data/mlmodels/kmeans/2D"
model.write().overwrite().save(kmeans_model_path)
model.__class__

## Get Predictions

In [ ]:
from pyspark.ml.clustering import KMeansModel
k_model = KMeansModel.load(kmeans_model_path)
predictions = k_model.transform(assembled_df)
predictions.show()

## Evaluate model

In [ ]:
from pyspark.ml.evaluation import ClusteringEvaluator

kmeans = KMeans().setK(3).setSeed(73)
model = kmeans.fit(assembled_df)
predictions = model.transform(assembled_df)
predictions.show()
# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score: {silhouette}")

# Show the result
print("Cluster Centers: ")
for center in model.clusterCenters():
    print(center)

# Lab 13: Clustering Wine dataset with K-means

In [ ]:
# Downlod dataset from https://www.kaggle.com/datasets/harrywang/wine-dataset-for-clustering

columns_types = [("Alcohol", "float"),
                                     ("Malic_Acid", "float"),
                                     ("Ash", "float"),
                                     ("Ash_Alcanity", "float"),
                                     ("Magnesium", "float"),
                                     ("Total_Phenols", "float"),
                                     ("Flavanoids", "float"),
                                     ("Nonflavanoid_Phenols", "float"),
                                     ("Proanthocyanins", "float"),
                                     ("Color_Intensity", "float"),
                                     ("Hue", "float"),
                                     ("OD280", "float"),
                                     ("Proline", "float")]

# Define schema for the DataFrame
wines_schema = SparkUtils.generate_schema(columns_types)

# Create DataFrame from wines csv
wines_df = su.spark \
                    .read \
                    .option("header", "true") \
                    .schema(wines_schema) \
                    .csv("/opt/spark/work-dir/data/ml/kmeans")


assembler = VectorAssembler(inputCols=[x for x,_ in columns_types], outputCol="features")
assembled_df = assembler.transform(wines_df)

# TODO: Find the optimial K
# TODO: Add the code here to iterate from k = 2, 4, .., 10 and get the silhouette score for each k

k = 2
kmeans = KMeans().setK(k).setSeed(13)

model = kmeans.fit(assembled_df)
print(f"K-means model trained successfully for {k} clusters")

# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()
predictions = model.transform(assembled_df)
silhouette = evaluator.evaluate(predictions)
print(f"Silhouette score: {silhouette}")


In [ ]:
su.spark.stop()